In [ ]:
!uv run python -m ipykernel install --user --name llm-zoomcamp-onnx --display-name "llm-zoomcamp-onnx"

In [ ]:
""" 
download the model 

models/
  Xenova/
    all-MiniLM-L6-v2/
      tokenizer.json
      model.onnx


available models:

Xenova/all-MiniLM-L6-v2 (384d) - best small general-purpose
Xenova/all-MiniLM-L12-v2 (384d) - better quality, slower
Xenova/paraphrase-MiniLM-L6-v2 (384d) - paraphrase detection
Xenova/paraphrase-multilingual-MiniLM-L12-v2 (384d) - multilingual
Xenova/multilingual-e5-small (384d) - multilingual retrieval
Xenova/multilingual-e5-base (768d) - stronger multilingual
Xenova/bge-small-en-v1.5 (384d) - strong retrieval
Xenova/bge-base-en-v1.5 (768d) - stronger retrieval
Xenova/gte-small (384d) - lightweight modern model
Xenova/gte-base (768d) - stronger GTE

To use a different model, add it to download.py, run the download, then update the path:
embed = Embedder("models/Xenova/bge-base-en-v1.5")
vectors = embed.encode("your text here")
print(vectors.shape)

"""
!uv run python download.py

## The Embedder class

1. Tokenize - convert text into integer IDs and attention masks
2. Run ONNX model - execute the model graph on CPU
3. Mean pooling - average the token embeddings, weighted by the attention mask
4. Normalize - divide by L2 norm so vectors can be compared with dot product

In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)


In [2]:
v1.dot(dv)

np.float64(0.32332403170761753)

In [3]:
v2.dot(dv)

np.float64(0.01973043944798833)

In [4]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-07-13 19:09:16--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     738  --.-KB/s    in 0s      

2026-07-13 19:09:18 (11.9 MB/s) - ‘ingest.py’ saved [738/738]



In [7]:
!uv add --active requests

Resolved 79 packages in 4.81s                                        
Prepared 1 package in 1.99s                                              
Installed 3 packages in 10ms.4.9                            
 + charset-normalizer==3.4.9
 + requests==2.34.2
 + urllib3==2.7.0


In [8]:
from ingest import load_faq_data
import requests
documents = load_faq_data()

In [9]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [10]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

/home/hsu/Documents/Learning/llm-zoomcamp-hw/llm-zoomcamp-onnx/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|███████████████████████████████████████████████████████████████████████████████████████████| 28/28 [00:25<00:00,  1.10it/s]


In [11]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}